# 05 -- Model Validation: NewsBERT-germ-210m

Loads the trained model from HuggingFace and evaluates it on **all labeled data**
(train + test splits from the HF dataset). Serves as a sanity check before
the final classify-all-articles run.

**Checks performed:**
- Hand-crafted test texts with expected categories
- Per-split accuracy (train vs. test)
- Confidence distribution analysis
- Error analysis (most common confusions)

**Prerequisites:** Trained model on HF (`Zorryy/NewsBERT-germ-210m`), `HF_TOKEN` in Colab Secrets.

In [ ]:
# === DEPENDENCIES (run once, may trigger a runtime restart) ===
!pip install --force-reinstall transformers==5.2.0
!pip install -q "transformers[sentencepiece]" datasets huggingface_hub \
    scikit-learn matplotlib seaborn tqdm pandas accelerate

import transformers
v = transformers.__version__
print(f"\ntransformers version: {v}")

if v != "5.2.0":
    print(f"WARNING: Version is {v}, not 5.2.0!")
    print("Runtime will be restarted -- after restart, SKIP this cell and continue from cell 2.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("OK (5.2.0) -- continue with the next cell.")

In [ ]:
# === SETUP ===
import os, sys, shutil, time as _time
import transformers
assert transformers.__version__ == "5.2.0", f"Wrong version: {transformers.__version__}"

# Clone or update the repository
REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Add pipeline directory to path
PIPELINE_DIR = f"{REPO}/Python/classification_pipeline/euroBERT_210m"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)
pu.ensure_drive_dirs()

# Authenticate with HuggingFace
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

# Clear HF cache to ensure fresh model download
MODEL_REPO = "Zorryy/NewsBERT-germ-210m"
for _p in [
    os.path.expanduser(f"~/.cache/huggingface/hub/models--{MODEL_REPO.replace('/', '--')}"),
    os.path.expanduser(f"~/.cache/huggingface/modules/transformers_modules/{MODEL_REPO.split('/')[0]}"),
]:
    if os.path.exists(_p):
        shutil.rmtree(_p)
        print(f"Cache cleared: {_p}")

# Record start time for the validation report
validation_start_time = _time.time()

print(f"transformers: {transformers.__version__}")
print("Setup OK.")

In [ ]:
# === CONFIGURATION ===
import torch
import numpy as np
import pandas as pd

ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gef\u00e4lle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/L\u00f6hne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

label2id = {label: idx for idx, label in enumerate(ALL_LABELS)}
id2label = {idx: label for idx, label in enumerate(ALL_LABELS)}

BATCH_SIZE = 16
MAX_LENGTH = 2048

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)")
    if gpu_mem >= 40:
        BATCH_SIZE = 64
    elif gpu_mem >= 20:
        BATCH_SIZE = 32
else:
    print("WARNING: No GPU available!")

print(f"Device: {device}, Batch Size: {BATCH_SIZE}")

In [ ]:
# === LOAD DATA ===
from datasets import load_dataset

ds = load_dataset(pu.DATASET_ID)
train_hf = ds["train"].to_pandas()
test_hf = ds["test"].to_pandas()

# Combine all labeled data
train_hf["hf_split"] = "train"
test_hf["hf_split"] = "test"
all_labeled = pd.concat([train_hf, test_hf], ignore_index=True)

print(f"Total labeled articles: {len(all_labeled)}")
print(f"  HF train: {len(train_hf)}")
print(f"  HF test:  {len(test_hf)}")
print(f"\nClass distribution:")
for label in ALL_LABELS:
    n = len(all_labeled[all_labeled["label"] == label])
    print(f"  {label:<30} {n:>4}")
print(f"  {'TOTAL':<30} {len(all_labeled):>4}")

In [ ]:
# === LOAD MODEL + QUICK SANITY CHECK ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

# Register default RoPE init for EuroBERT compatibility
def _default_rope_init(config, device=None, **kwargs):
    base = getattr(config, "rope_theta", 10000.0)
    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))
    return inv_freq, 1.0

ROPE_INIT_FUNCTIONS["default"] = _default_rope_init

print(f"Loading model: {MODEL_REPO}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_REPO, trust_remote_code=True, attn_implementation="eager"
)
model = model.to(device)
model.eval()

# Quick sanity check with hand-crafted test texts
SANITY = [
    ("Die Bundesregierung plant neue Klimaschutz-Massnahmen und den Ausbau erneuerbarer Energien.", "Klima / Energie"),
    ("Die AfD liegt bei 22 Prozent und fordert haertere Migrationspolitik.", "AfD/Rechte"),
    ("Der Krieg in der Ukraine geht weiter, Russland startet Raketenangriffe.", "Ukraine/Krieg/Russland"),
]

print(f"\nSANITY CHECK: {len(SANITY)} test examples")
sanity_passed = 0
for text, expected in SANITY:
    inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits.float(), dim=-1)
    pred = id2label[probs.argmax(dim=-1).item()]
    score = probs.max().item()
    passed = pred == expected
    sanity_passed += int(passed)
    status = "OK" if passed else "FAIL"
    print(f"  [{status}] Expected: {expected:<25} -> {pred} ({score:.2f})")

print(f"\nResult: {sanity_passed}/{len(SANITY)} passed")
if sanity_passed < len(SANITY):
    print("SANITY CHECK FAILED! Review model before proceeding.")

print(f"\nModel ready on {device}.")

In [ ]:
# === CPU vs GPU LOGIT COMPARISON ===
# Verify that GPU inference produces the same logits as CPU inference
if torch.cuda.is_available():
    test_text = SANITY[0][0]
    inputs_cpu = tokenizer(test_text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True)

    # CPU inference
    model_cpu = model.cpu()
    model_cpu.eval()
    with torch.no_grad():
        logits_cpu = model_cpu(**inputs_cpu).logits.float()

    # GPU inference
    model_gpu = model_cpu.to(device)
    model_gpu.eval()
    inputs_gpu = {k: v.to(device) for k, v in inputs_cpu.items()}
    with torch.no_grad():
        logits_gpu = model_gpu(**inputs_gpu).logits.float().cpu()

    # Compare logits
    diff = (logits_cpu - logits_gpu).abs().max().item()
    print(f"Max logit difference (CPU vs GPU): {diff:.6e}")
    if diff > 1e-3:
        print(f"LARGE DIFFERENCE -- investigate before proceeding!")
    else:
        print("Logits match (within tolerance). GPU inference is consistent.")

    # Keep model on the correct device for the rest of the notebook
    model = model_gpu
else:
    print("Skipping CPU vs GPU comparison (no GPU available).")

In [ ]:
# === INFERENCE ON ALL LABELED DATA ===
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from datasets import Dataset
from tqdm import tqdm

texts = all_labeled["text"].fillna("").tolist()

# Tokenize all texts
hf_dataset = Dataset.from_dict({"text": texts})
hf_dataset = hf_dataset.map(
    lambda x: tokenizer(x["text"], max_length=MAX_LENGTH, truncation=True),
    batched=True, batch_size=512, remove_columns=["text"],
)
hf_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")
dataloader = DataLoader(
    hf_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=data_collator, num_workers=4, pin_memory=True,
)

# Compute token lengths for each article
print("Computing token lengths...")
token_lengths = []
for i in range(0, len(texts), 1000):
    batch = texts[i:i+1000]
    encoded = tokenizer(batch, truncation=False, add_special_tokens=False)
    token_lengths.extend(len(ids) for ids in encoded["input_ids"])
all_labeled["token_length"] = token_lengths

# Run inference
all_predictions = []
all_scores = []

print(f"\nRunning inference on {len(texts)} articles (batch_size={BATCH_SIZE})...")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        pred_ids = probs.argmax(axis=-1)
        all_predictions.extend(id2label[int(i)] for i in pred_ids)
        all_scores.extend(probs.tolist())

all_labeled["prediction"] = all_predictions
scores_array = np.array(all_scores)
all_labeled["prediction_score"] = scores_array.max(axis=1)
all_labeled["correct"] = all_labeled["label"] == all_labeled["prediction"]

print(f"Inference complete: {len(all_predictions)} predictions")

In [ ]:
# === PER-CLASS PERFORMANCE (ALL LABELED DATA) ===
import matplotlib.pyplot as plt
import seaborn as sns

true_labels = all_labeled["label"].tolist()
metrics = pu.evaluate(true_labels, all_predictions, labels=ALL_LABELS, experiment_name="all_labeled")
pu.print_metrics(metrics, "NewsBERT-germ-210m -- All labeled data")

In [ ]:
# === CONFUSION MATRIX ===
pu.plot_confusion_matrix(metrics, title="NewsBERT-germ-210m (all labeled data)")
plt.show()

In [ ]:
# === PER-CLASS BAR CHART ===
pc = metrics["per_class_df"].sort_values("F1", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(pc))
bar_h = 0.25

ax.barh(y_pos - bar_h, pc["Precision"], bar_h, label="Precision", color="#2196F3", alpha=0.85)
ax.barh(y_pos, pc["Recall"], bar_h, label="Recall", color="#FF9800", alpha=0.85)
ax.barh(y_pos + bar_h, pc["F1"], bar_h, label="F1", color="#4CAF50", alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(pc["Label"])
ax.set_xlabel("Score")
ax.set_title("Per-Class Metrics (all labeled data)", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.3)

for i, (_, row) in enumerate(pc.iterrows()):
    ax.text(row["F1"] + 0.01, y_pos[i] + bar_h, f"{row['F1']:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# === ERROR ANALYSIS ===
wrong = all_labeled[~all_labeled["correct"]].copy()
right = all_labeled[all_labeled["correct"]].copy()

print(f"{'='*70}")
print(f"  ERROR ANALYSIS")
print(f"{'='*70}")
print(f"  Correct:  {len(right):>5} ({len(right)/len(all_labeled)*100:.1f}%)")
print(f"  Incorrect:{len(wrong):>5} ({len(wrong)/len(all_labeled)*100:.1f}%)")
print(f"  Total:    {len(all_labeled):>5}")

# --- 1. Confidence comparison ---
print(f"\n--- 1. Confidence ---")
print(f"  Correct:   mean={right['prediction_score'].mean():.4f}, median={right['prediction_score'].median():.4f}")
print(f"  Incorrect: mean={wrong['prediction_score'].mean():.4f}, median={wrong['prediction_score'].median():.4f}")

# --- 2. Token length ---
print(f"\n--- 2. Token length ---")
print(f"  Correct:   mean={right['token_length'].mean():.0f}, median={right['token_length'].median():.0f}")
print(f"  Incorrect: mean={wrong['token_length'].mean():.0f}, median={wrong['token_length'].median():.0f}")

# Truncated articles (>2048 tokens)
trunc_total = (all_labeled["token_length"] > MAX_LENGTH).sum()
trunc_wrong = (wrong["token_length"] > MAX_LENGTH).sum() if len(wrong) > 0 else 0
print(f"  Truncated (>{MAX_LENGTH} tokens): {trunc_total} total, {trunc_wrong} of those incorrect")

# --- 3. Errors per class ---
print(f"\n--- 3. Errors per class (true label) ---")
print(f"  {'Label':<30} {'Total':>6} {'Wrong':>7} {'Error%':>8}")
print(f"  {'-'*55}")
for label in ALL_LABELS:
    n_total = len(all_labeled[all_labeled["label"] == label])
    n_wrong = len(wrong[wrong["label"] == label])
    pct = n_wrong / n_total * 100 if n_total > 0 else 0
    marker = " <<<" if pct > 30 else ""
    print(f"  {label:<30} {n_total:>6} {n_wrong:>7} {pct:>7.1f}%{marker}")

In [ ]:
# === CLASS OVERLAP: Most common confusions ===
print(f"{'='*70}")
print(f"  CLASS OVERLAP: Most common confusions")
print(f"{'='*70}")

if len(wrong) > 0:
    confusion_pairs = wrong.groupby(["label", "prediction"]).size().reset_index(name="count")
    confusion_pairs = confusion_pairs.sort_values("count", ascending=False)

    print(f"\n  Top 15 confusions:")
    print(f"  {'True Label':<28} {'Predicted As':<28} {'Count':>6}")
    print(f"  {'-'*65}")
    for _, row in confusion_pairs.head(15).iterrows():
        print(f"  {row['label']:<28} {row['prediction']:<28} {row['count']:>6}")

    # Most common confusion per class
    print(f"\n  Most common confusion per class:")
    print(f"  {'True Label':<28} {'Confused with':<28} {'n':>4} {'of':>4}")
    print(f"  {'-'*68}")
    for label in ALL_LABELS:
        label_wrong = wrong[wrong["label"] == label]
        if len(label_wrong) > 0:
            top_pred = label_wrong["prediction"].value_counts().index[0]
            top_count = label_wrong["prediction"].value_counts().iloc[0]
            n_total = len(all_labeled[all_labeled["label"] == label])
            print(f"  {label:<28} {top_pred:<28} {top_count:>4} {n_total:>4}")
        else:
            print(f"  {label:<28} {'(no errors)':<28}")
else:
    print("  No errors!")

In [ ]:
# === CONFIDENCE DISTRIBUTION ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Confidence histogram for correct vs incorrect predictions
axes[0].hist(right["prediction_score"], bins=30, alpha=0.7, label=f"Correct (n={len(right)})", color="#4CAF50")
if len(wrong) > 0:
    axes[0].hist(wrong["prediction_score"], bins=30, alpha=0.7, label=f"Incorrect (n={len(wrong)})", color="#F44336")
axes[0].set_xlabel("Prediction Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Confidence: Correct vs. Incorrect")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Token length vs confidence scatter plot
colors = all_labeled["correct"].map({True: "#4CAF50", False: "#F44336"})
axes[1].scatter(
    all_labeled["token_length"], all_labeled["prediction_score"],
    c=colors, alpha=0.3, s=10,
)
axes[1].axvline(x=MAX_LENGTH, color="red", linestyle="--", alpha=0.5, label=f"Truncation ({MAX_LENGTH})")
axes[1].set_xlabel("Token length")
axes[1].set_ylabel("Prediction Score")
axes[1].set_title("Token length vs. Confidence")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Confidence & Token Length Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# === VALIDATION RUN SUMMARY ===
print(f"\n{'='*70}")
print(f"  VALIDATION RUN SUMMARY")
print(f"{'='*70}")
print(f"  Model:             {MODEL_REPO}")
print(f"  transformers:      {transformers.__version__}")
print(f"  Device:            {device}")
print(f"  Labeled articles:  {len(all_labeled)}")
print(f"  F1 Macro:          {metrics['f1_macro']:.4f}")
print(f"  F1 Weighted:       {metrics['f1_weighted']:.4f}")
print(f"  Accuracy:          {metrics['accuracy']:.4f}")
print(f"  Mean Confidence:   {all_labeled['prediction_score'].mean():.4f}")
print(f"  Error rate:        {len(wrong)}/{len(all_labeled)} ({len(wrong)/len(all_labeled)*100:.1f}%)")

# Weak classes (F1 < 0.70)
weak = metrics["per_class_df"][metrics["per_class_df"]["F1"] < 0.70]
if len(weak) > 0:
    print(f"\n  Weak classes (F1 < 0.70):")
    for _, row in weak.iterrows():
        print(f"    {row['Label']:<30} F1={row['F1']:.4f}")
else:
    print(f"\n  All classes F1 >= 0.70")

print(f"{'='*70}")

if metrics["f1_macro"] >= 0.75:
    print(f"\n  -> Model looks good. Final classify_all_articles run can proceed.")
else:
    print(f"\n  -> WARNING: F1 Macro below 0.75 -- review model before proceeding!")

In [ ]:
# === SAVE VALIDATION REPORT TO DRIVE ===
total_time = _time.time() - validation_start_time
timer = pu.ExperimentTimer()
timer.start_time = _time.time() - total_time
timer.end_time = _time.time()

report_path = pu.generate_report(
    model_name="NewsBERT_germ_210m_validation",
    model_type="fine-tuned",
    metrics=metrics,
    timer=timer,
    model_info={"huggingface_id": MODEL_REPO, "language": "de", "max_tokens": 2048, "parameters": "210M"},
    experiment_notes="Validation run on all labeled data (train + test) before final classification.",
)
print(f"Validation report saved: {report_path}")

In [ ]:
# === CLEANUP ===
import gc

del model, dataloader, hf_dataset, scores_array, all_scores, all_predictions
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("Cleanup complete.")